In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime
import time

# ================= ERROR =================
def show_clean_error(e, step="Unknown"):
    messagebox.showerror(
        "Error",
        f"Error: {type(e).__name__}\nStep: {step}\nMessage: {str(e)}"
    )

# ================= GOOGLE =================
CREDS_FILE = "creds.json"

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

BRAND_SHEET_MAP = {
    "durex": "Durex CRM 2.0_Plan Performance",
    "neurogum": "Neurogum CRM Tracker",
    "avon": "Avon India Campaign Planning",
    "enamor": "Enamor Campaign Planning",
    
}

# ================= FILE =================
def pick_file(entry):
    path = filedialog.askopenfilename()

    entry.delete(0, tk.END)
    entry.insert(0, path)

def load_file(path):

    if not path:
        return None

    if path.lower().endswith(".csv"):
        return pd.read_csv(path, low_memory=False)

    return pd.read_excel(path)

# ================= HELPERS =================
def col_num_to_letter(n):

    result = ""

    while n > 0:
        n, rem = divmod(n - 1, 26)
        result = chr(65 + rem) + result

    return result

# ================= ORDERS =================
def get_orders_revenue(df, utm):

    if df is None:
        return 0, 0

    df.columns = df.columns.str.strip()

    utm_col = "Utm Campaign"
    orders_col = "Total Qty Ordered"
    revenue_col = "Grand Total"

    df[utm_col] = (
        df[utm_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    f = df[df[utm_col] == utm]

    orders = f[orders_col].notna().sum()

    revenue = pd.to_numeric(
        f[revenue_col],
        errors="coerce"
    ).sum()

    return int(orders), round(revenue, 1)

# ================= SESSIONS =================
def get_sessions(df, utm):

    if df is None:
        return 0

    utm_col = next(
        (c for c in df.columns if "utm" in c.lower()),
        None
    )

    ses_col = next(
        (c for c in df.columns if "session" in c.lower()),
        None
    )

    if not utm_col or not ses_col:
        return 0

    df[utm_col] = (
        df[utm_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return round(
        pd.to_numeric(
            df[df[utm_col] == utm][ses_col],
            errors="coerce"
        ).sum(),
        1
    )

# ================= UPDATE SHEET =================
def update_sheet(sheet, utm, data):

    df = pd.DataFrame(sheet.get_all_records())

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    df["utm name used"] = (
        df["utm name used"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    header = df.columns.tolist()

    if utm in df["utm name used"].values:

        idx = df.index[
            df["utm name used"] == utm
        ][0] + 2

        row = df.iloc[idx - 2].tolist()

        for k, v in data.items():

            if k in header:
                row[header.index(k)] = str(v)

        end_col = col_num_to_letter(len(header))

        sheet.update(
            f"A{idx}:{end_col}{idx}",
            [row]
        )

    else:

        row = [""] * len(header)

        for k, v in data.items():

            if k in header:
                row[header.index(k)] = str(v)

        sheet.append_row(row)

# ================= CALCULATE METRICS =================
def calculate_metrics(
    sent,
    delivered,
    opens,
    clicks,
    spends,
    orders,
    revenue,
    sessions
):

    data = {}

    data["delivery rate"] = (
        f"{(delivered/sent*100):.1f}%"
        if sent else "0.0%"
    )

    data["open rate"] = (
        f"{(opens/delivered*100):.1f}%"
        if delivered else "0.0%"
    )

    data["click rate"] = (
        f"{(clicks/delivered*100):.1f}%"
        if delivered else "0.0%"
    )

    data["ctor"] = (
        f"{(clicks/opens*100):.1f}%"
        if opens else "0.0%"
    )

    data["cvr"] = (
        f"{(orders/sessions*100):.1f}%"
        if sessions else "0.0%"
    )

    data["sessions to clicks rate"] = (
        f"{(sessions/clicks*100):.1f}%"
        if clicks else "0.0%"
    )

    data["roas"] = (
        round(revenue/spends, 1)
        if spends else 0.0
    )

    data["aov"] = (
        round(revenue/orders, 1)
        if orders else 0.0
    )

    return data

# ================= SINGLE =================
def process_single(
    sheet,
    utm,
    kwik_df,
    orders_df,
    sessions_df
):

    utm = utm.strip().lower()

    kwik_df.columns = kwik_df.columns.str.strip()

    col_map = {
        c.lower(): c
        for c in kwik_df.columns
    }

    sent = delivered = opens = clicks = spends = 0
    orders = revenue = sessions = 0

    # ================= KWIK =================
    if kwik_var.get() and kwik_df is not None:

        sent = len(kwik_df)

        delivered = kwik_df[
            kwik_df[col_map["status"]]
            .isin(["Delivered", "Seen"])
        ].shape[0]

        opens = kwik_df[
            kwik_df[col_map["status"]] == "Seen"
        ].shape[0]

        clicks = kwik_df[
            col_map["clicked at"]
        ].notna().sum()

        spends = round(delivered * 0.9, 1)

    # ================= ORDERS =================
    if orders_var.get():

        orders, revenue = get_orders_revenue(
            orders_df,
            utm
        )

    # ================= SESSIONS =================
    if sessions_var.get():

        sessions = get_sessions(
            sessions_df,
            utm
        )

    now = datetime.now()

    data = {
        "utm name used": utm,
        "last updated date": now.strftime("%d-%B-%Y"),
        "last updated day": now.strftime("%a"),
        "last updated time": now.strftime("%I:%M %p"),
    }

    # ================= RAW METRICS =================
    if kwik_var.get():

        data.update({
            "total sent": round(sent, 1),
            "delivered": round(delivered, 1),
            "opens": round(opens, 1),
            "clicks": round(clicks, 1),
            "spends": round(spends, 1),
        })

    if orders_var.get():

        data.update({
            "orders": round(orders, 1),
            "revenue": round(revenue, 1),
        })

    if sessions_var.get():

        data.update({
            "sessions": round(sessions, 1),
        })

    # ================= DERIVED METRICS =================
    metrics = calculate_metrics(
        sent,
        delivered,
        opens,
        clicks,
        spends,
        orders,
        revenue,
        sessions
    )

    data.update(metrics)

    update_sheet(sheet, utm, data)

# ================= BULK =================
def process_bulk(
    sheet,
    kwik_df,
    orders_df,
    sessions_df
):

    kwik_df.columns = kwik_df.columns.str.strip()

    utm_col = next(
        (
            c for c in kwik_df.columns
            if c.lower() in ["name", "campaign name"]
        ),
        None
    )

    utm_list = (
        kwik_df[utm_col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

    for utm in utm_list:

        try:

            time.sleep(0.7)

            temp = kwik_df[
                kwik_df[utm_col]
                .astype(str)
                .str.strip()
                .str.lower() == utm
            ]

            sent = delivered = opens = clicks = spends = 0
            orders = revenue = sessions = 0

            # ================= KWIK =================
            if kwik_var.get():

                sent = (
                    temp["Sent"].sum()
                    if "Sent" in temp.columns
                    else 0
                )

                delivered = (
                    temp["Delivered"].sum()
                    if "Delivered" in temp.columns
                    else 0
                )

                opens = (
                    temp["Seen"].sum()
                    if "Seen" in temp.columns
                    else 0
                )

                clicks = (
                    temp["Clicks"].sum()
                    if "Clicks" in temp.columns
                    else 0
                )

                spends = round(delivered * 0.9, 1)

            # ================= ORDERS =================
            if orders_var.get():

                orders, revenue = get_orders_revenue(
                    orders_df,
                    utm
                )

            # ================= SESSIONS =================
            if sessions_var.get():

                sessions = get_sessions(
                    sessions_df,
                    utm
                )

            now = datetime.now()

            data = {
                "utm name used": utm,
                "last updated date": now.strftime("%d-%B-%Y"),
                "last updated day": now.strftime("%a"),
                "last updated time": now.strftime("%I:%M %p"),
            }

            # ================= RAW METRICS =================
            if kwik_var.get():

                data.update({
                    "total sent": round(sent, 1),
                    "delivered": round(delivered, 1),
                    "opens": round(opens, 1),
                    "clicks": round(clicks, 1),
                    "spends": round(spends, 1),
                })

            if orders_var.get():

                data.update({
                    "orders": round(orders, 1),
                    "revenue": round(revenue, 1),
                })

            if sessions_var.get():

                data.update({
                    "sessions": round(sessions, 1),
                })

            # ================= DERIVED METRICS =================
            metrics = calculate_metrics(
                sent,
                delivered,
                opens,
                clicks,
                spends,
                orders,
                revenue,
                sessions
            )

            data.update(metrics)

            update_sheet(sheet, utm, data)

        except Exception as e:

            show_clean_error(
                e,
                f"Bulk UTM: {utm}"
            )

# ================= RUN =================
def run():

    try:

        global client

        creds = (
            ServiceAccountCredentials
            .from_json_keyfile_name(
                CREDS_FILE,
                scope
            )
        )

        client = gspread.authorize(creds)

        brand = (
            brand_dropdown
            .get()
            .strip()
            .lower()
        )

        sheet_name = sheet_entry.get().strip()

        sheet = (
            client
            .open(BRAND_SHEET_MAP[brand])
            .worksheet(sheet_name)
        )

        kwik = load_file(kwik_entry.get())

        orders = (
            load_file(orders_entry.get())
            if orders_var.get()
            else None
        )

        sessions = (
            load_file(sessions_entry.get())
            if sessions_var.get()
            else None
        )

        if mode.get() == "single":

            process_single(
                sheet,
                utm_entry.get(),
                kwik,
                orders,
                sessions
            )

        else:

            process_bulk(
                sheet,
                kwik,
                orders,
                sessions
            )

        messagebox.showinfo(
            "Done",
            "Updated Successfully"
        )

    except Exception as e:

        show_clean_error(e, "Run")

# ================= UI =================
root = tk.Tk()

root.title("CRM Tool")
root.geometry("500x700")

# ================= BRAND =================
tk.Label(root, text="Brand").pack()

brand_dropdown = ttk.Combobox(
    root,
    values=list(BRAND_SHEET_MAP.keys())
)

brand_dropdown.pack()

# ================= SHEET =================
tk.Label(
    root,
    text="Tab Name (in the sheet to be updated)"
).pack()

sheet_entry = tk.Entry(root)
sheet_entry.pack()

# ================= FILE BLOCK =================
def file_block(label):

    tk.Label(root, text=label).pack()

    e = tk.Entry(root)
    e.pack()

    tk.Button(
        root,
        text="Browse",
        command=lambda: pick_file(e)
    ).pack()

    return e

# ================= FILES =================
kwik_entry = file_block("Kwikengage File")

kwik_var = tk.BooleanVar(value=True)
orders_var = tk.BooleanVar(value=True)
sessions_var = tk.BooleanVar(value=True)

tk.Checkbutton(
    root,
    text="Update Kwikengage",
    variable=kwik_var
).pack()

orders_entry = file_block("Orders File")

tk.Checkbutton(
    root,
    text="Update Orders",
    variable=orders_var
).pack()

sessions_entry = file_block("Sessions File")

tk.Checkbutton(
    root,
    text="Update Sessions",
    variable=sessions_var
).pack()

# ================= MODE =================
mode = tk.StringVar(value="single")

tk.Radiobutton(
    root,
    text="Single UTM",
    variable=mode,
    value="single"
).pack()

tk.Radiobutton(
    root,
    text="Bulk UTM",
    variable=mode,
    value="bulk"
).pack()

# ================= SINGLE UTM =================
tk.Label(root, text="UTM (for single)").pack()

utm_entry = tk.Entry(root)
utm_entry.pack()

# ================= RUN =================
tk.Button(
    root,
    text="Run",
    command=run,
    bg="green",
    fg="white"
).pack(pady=20)

root.mainloop()
